## Automatic annotation using LocateAnything text-prompt model

In [10]:
import json
import sys
from pathlib import Path
from PIL import Image
from tqdm import tqdm


# Also add src/ for locateanything_worker import
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [11]:
import os 
orig = Path.cwd()
os.chdir("../weights")

from locateanything_worker import LocateAnythingWorker

worker = LocateAnythingWorker("LocateAnything-3B")
print("✅ LocateAnything model loaded")

os.chdir(orig)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/kewei/anaconda3/envs/auto_annotation/lib/python3.12/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
flash_attn is not available for MoonViT inference; falling back to sdpa.
Qwen2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `Genera

✅ LocateAnything model loaded


### Configure class → text prompt mapping

In [18]:
# class_id → (Chinese label, English text prompt for LocateAnything)
class_config = {
    0: ("red box", "red box"),
    1: ("small object", "small object"),
    2: ("green small box", "green small box"),
    3: ("toy car", "toy car"),
}
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT = PROJECT_ROOT.parent
print(PROJECT_ROOT)

target_folder = PROJECT_ROOT / "datasets/giftbox_task/images"
annotation_save_path = PROJECT_ROOT / "datasets/giftbox_task/annotations/annotations.json"

target_paths = sorted(target_folder.glob("*"))
print(f"Found {len(target_paths)} target images")
print(f"Classes: {[v[0] for v in class_config.values()]}")

/home/kewei/automatic-annotation
Found 228 target images
Classes: ['red box', 'small object', 'green small box', 'toy car']


## Run LocateAnything text-prompt inference & save COCO annotations

In [19]:
import re


def bbox_to_polygon(x1, y1, x2, y2):
    """Convert bbox [x1,y1,x2,y2] to a rectangle polygon for COCO segmentation."""
    return [x1, y1, x2, y1, x2, y2, x1, y2]


def locany_predict_bboxes(worker, image, prompts):
    """
    Run LocateAny detect on a single image, return parsed bboxes.
    Returns list of (label_name, x1, y1, x2, y2).
    """
    result = worker.detect(image, prompts)
    answer = result["answer"]
    w, h = image.size

    # Parse both label + box coordinates from answer
    pattern = r"<ref>(.*?)</ref><box><(\d+)><(\d+)><(\d+)><(\d+)></box>"
    matches = re.findall(pattern, answer)

    results = []
    for label, x1_raw, y1_raw, x2_raw, y2_raw in matches:
        x1 = float(x1_raw) / 1000 * w
        y1 = float(y1_raw) / 1000 * h
        x2 = float(x2_raw) / 1000 * w
        y2 = float(y2_raw) / 1000 * h
        results.append((label, x1, y1, x2, y2))

    return results


def make_coco_annotation_locany(target_paths, class_config, worker, save_path, min_area=0):
    """
    Run LocateAny on all images, one detect call per image, save COCO JSON.
    """
    coco_output = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": cid, "name": label, "supercategory": "object"}
            for cid, (label, _) in class_config.items()
        ],
    }

    # Build label → class_id reverse mapping
    label_to_cid = {prompt: cid for cid, (label, prompt) in class_config.items()}
    all_prompts = [prompt for label, prompt in class_config.values()]

    ann_id = 1
    total_objs = 0

    # Build images list
    for img_id, target_path in enumerate(target_paths, start=1):
        target_img = Image.open(target_path).convert("RGB")
        w, h = target_img.size
        coco_output["images"].append({
            "id": img_id,
            "file_name": target_path.name,
            "width": w,
            "height": h,
        })

    # Run inference: per image, one call with all prompts
    for img_id, target_path in enumerate(tqdm(target_paths, desc="LocateAny inference"), start=1):
        image = Image.open(target_path).convert("RGB")
        bboxes = locany_predict_bboxes(worker, image, all_prompts)

        for label_name, x1, y1, x2, y2 in bboxes:
            bbox_w = x2 - x1
            bbox_h = y2 - y1
            area = bbox_w * bbox_h
            if area < min_area:
                continue
            if label_name not in label_to_cid:
                continue

            seg = bbox_to_polygon(x1, y1, x2, y2)
            coco_output["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": label_to_cid[label_name],
                "segmentation": [seg],
                "bbox": [float(x1), float(y1), float(bbox_w), float(bbox_h)],
                "area": float(area),
                "iscrowd": 0,
            })
            ann_id += 1
            total_objs += 1

    # Save JSON
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(coco_output, f, indent=2, ensure_ascii=False)

    # Save labels.txt
    labels_path = save_path.parent / "labels.txt"
    with open(labels_path, "w", encoding="utf-8") as f_labels:
        for cid in sorted(class_config.keys()):
            f_labels.write(f"{class_config[cid][0]}\n")

    print(f"\n✅ COCO annotation saved to {save_path}")
    print(f"   images={len(coco_output['images'])}, "
          f"annotations={len(coco_output['annotations'])}, "
          f"categories={len(coco_output['categories'])}")
    print(f"   total objects found: {total_objs}")

In [20]:
# Run auto-annotation
if not annotation_save_path.exists():
    annotation_save_path.parent.mkdir(parents=True, exist_ok=True)
    annotation_save_path.touch()

make_coco_annotation_locany(
    target_paths=target_paths,
    class_config=class_config,
    worker=worker,
    save_path=annotation_save_path,
    min_area=0,
)

LocateAny inference:   0%|          | 0/228 [00:00<?, ?it/s]/home/kewei/.cache/huggingface/modules/transformers_modules/LocateAnything_hyphen_3B/generate_utils.py:186: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  box_avg.append(torch.tensor(out_ref, dtype=x0.dtype, device=x0.device))
LocateAny inference:   0%|          | 1/228 [00:00<02:46,  1.36it/s]


Statistic Info, num_tokens=47; generate_time(s)=0.7105; tps=66.1472; forward_step=21; num_boxes=5; bps=7.0369; prefill_time=0.4330; switch_to_ar=2



LocateAny inference:   1%|          | 2/228 [00:00<01:43,  2.18it/s]


Statistic Info, num_tokens=50; generate_time(s)=0.2577; tps=194.0209; forward_step=11; num_boxes=5; bps=19.4021; prefill_time=0.1089; switch_to_ar=0



LocateAny inference:   1%|▏         | 3/228 [00:01<01:34,  2.39it/s]

...


Statistic Info, num_tokens=39; generate_time(s)=0.2898; tps=134.5530; forward_step=14; num_boxes=4; bps=13.8003; prefill_time=0.1097; switch_to_ar=1



LocateAny inference: 100%|█████████▉| 227/228 [01:12<00:00,  3.33it/s]


Statistic Info, num_tokens=42; generate_time(s)=0.2839; tps=147.9362; forward_step=14; num_boxes=4; bps=14.0892; prefill_time=0.1096; switch_to_ar=1



LocateAny inference: 100%|██████████| 228/228 [01:13<00:00,  3.11it/s]


Statistic Info, num_tokens=50; generate_time(s)=0.3138; tps=159.3625; forward_step=16; num_boxes=5; bps=15.9363; prefill_time=0.1095; switch_to_ar=1


✅ COCO annotation saved to /home/kewei/automatic-annotation/datasets/giftbox_task/annotations/annotations.json
   images=228, annotations=664, categories=4
   total objects found: 664
